# trace

In [ ]:
///- --test static

open testing

In [ ]:
open sm'_operators

## trace

### trace_level

In [ ]:
union trace_level =
    | Verbose
    | Debug
    | Info
    | Warning
    | Critical

### severity

In [ ]:
inl severity = function
    | Verbose => 10
    | Debug => 20
    | Info => 30
    | Warning => 40
    | Critical => 50

### read_state

In [ ]:
inl read_state () =
    run_target function
        | Zig => fun () =>
            {
                trace_level = "TRACE_LEVEL" |> env.get_environment_variable |> reflection.union_try_pick : option trace_level
                repl_start = None : option i64
            }
        | Rust => fun () =>
            {
                trace_level = "TRACE_LEVEL" |> env.get_environment_variable |> reflection.union_try_pick
                repl_start = None
            }
        | TypeScript => fun () =>
            {
                trace_level = "TRACE_LEVEL" |> env.get_environment_variable |> reflection.union_try_pick'
                repl_start = None
            }
        | _ => fun () =>
            join
                {
                    trace_level =
                        inl get_result () =
                            "TRACE_LEVEL"
                            |> env.get_environment_variable
                            |> reflection.union_try_pick
                        backend_switch {
                            Gleam = get_result : () -> option trace_level
                            Lua = fun () => None : option trace_level
                            Fsharp = get_result : () -> option trace_level
                            Python = get_result : () -> option trace_level
                        }
                    repl_start =
                        inl automation = env.get_environment_variable "AUTOMATION"
                        if automation <>. "True"
                        then None
                        else
                            date_time.now ()
                            |> date_time.to_timestamp
                            |> fun (date_time.timestamp x) => x
                            |> Some
                }

### trace_state

In [ ]:
type trace_state =
    {
        count : mut i64
        trace_file : mut (string -> ())
        enabled : mut bool
        acc : mut string
        level : mut trace_level
        repl_start : optionm'.option' i64
    }

### new_trace_state

In [ ]:
let new_trace_state trace_level' =
    inl { repl_start trace_level } = read_state ()
    {
        count = mut 1i64
        trace_file = mut ignore
        enabled = mut true
        acc = mut ""
        level = trace_level |> optionm'.default_value trace_level' |> mut
        repl_start = repl_start |> optionm'.box
    } : trace_state

### init_trace_state

In [ ]:
inl init_trace_state trace_level : () =
    inl trace_level = trace_level |> optionm'.default_value Verbose
    backend_switch {
        Gleam = fun () =>
            global "fn trace_state () { option.None }"
            $'let new_trace_state = !new_trace_state ' : ()
            $'let trace_level = !trace_level ' : ()
            $'let trace_state = fn () {' : ()
            $'  case trace_state () {' : ()
            $'    option.None -> new_trace_state(trace_level) |> option.Some' : ()
            $'    x -> x' : ()
            $'  }' : ()
            $'}' : ()
        Lua = fun () =>
            global "_G.trace_state = nil"
            $'if _G.trace_state == nil then _G.trace_state = !new_trace_state(!trace_level) end' : ()
        Fsharp = fun () =>
            backend_switch {
                Fsharp = fun () =>
                    global "module TraceState = let mutable trace_state = None"
            }
            fun () =>
                if $'TraceState.trace_state.IsNone' then
                    inl trace_state = new_trace_state trace_level |> optionm'.some'
                    $'TraceState.trace_state <- !trace_state ' : ()
            |> exec_unit
        Python = fun () =>
            global "class TraceState: trace_state = None"
            $'if TraceState.trace_state is None: TraceState.trace_state = !new_trace_state(!trace_level)' : ()
        TypeScript = fun () =>
            global "let spiral_trace_state: any = undefined;"
            $'if (spiral_trace_state === undefined) spiral_trace_state = !new_trace_state(!trace_level)' : ()
        Rust = fun () =>
            global """
                fn spiral_trace_hold<T: Clone>(fresh: &std::rc::Rc<dyn Fn() -> T>) -> T {
                    use std::sync::OnceLock;
                    static SLOT: OnceLock<usize> = OnceLock::new();
                    let raw = *SLOT.get_or_init(|| Box::into_raw(Box::new((*fresh)())) as usize);
                    unsafe { (*(raw as *const T)).clone() }
                }
                """
            inl fresh = fun () => new_trace_state trace_level
            open rust.rust_operators
            !\\(fresh, $'"{ let _ = spiral_trace_hold(&$0); }"') : ()
        Zig = fun () => ()
    }

### get_trace_state_or_init

In [ ]:
inl get_trace_state_or_init trace_level : trace_state =
    init_trace_state trace_level
    backend_switch {
        Gleam = fun () =>
            inl trace_level = trace_level |> optionm'.default_value Verbose
            $'trace_state () |> option.lazy_unwrap(fn () { !new_trace_state(!trace_level) })' : trace_state
        Lua = fun () =>
            $'_G.trace_state' : trace_state
        Fsharp = fun () =>
            $'TraceState.trace_state.Value' : trace_state
        Python = fun () =>
            $'TraceState.trace_state' : trace_state
        TypeScript = fun () =>
            $'spiral_trace_state' : trace_state
        Rust = fun () =>
            global """
                fn spiral_trace_hold<T: Clone>(fresh: &std::rc::Rc<dyn Fn() -> T>) -> T {
                    use std::sync::OnceLock;
                    static SLOT: OnceLock<usize> = OnceLock::new();
                    let raw = *SLOT.get_or_init(|| Box::into_raw(Box::new((*fresh)())) as usize);
                    unsafe { (*(raw as *const T)).clone() }
                }
                """
            inl level' = trace_level |> optionm'.default_value Verbose
            inl fresh = fun () => new_trace_state level'
            open rust.rust_operators
            !\\(fresh, $'"{ spiral_trace_hold(&$0) }"') : trace_state
        Zig = fun () =>
            global """
                fn spiralHold(comptime T: type, fresh: anytype) T {
                    const S = struct {
                        var slot: ?T = null;
                    };
                    if (S.slot == null) S.slot = fresh.call(fresh.ctx);
                    return S.slot.?;
                }
                """
            inl level' = trace_level |> optionm'.default_value Verbose
            inl fresh = dyn (fun () => new_trace_state level')
            $'spiralHold(`trace_state, !fresh)' : trace_state
    }

### set_trace_level

In [ ]:
inl set_trace_level new_level =
    inl ({ level } & trace_state) = get_trace_state_or_init None
    level <- new_level
    backend_switch {
        Gleam = fun () => $'let trace_state = fn () { !(trace_state |> optionm'.some') }' : ()
        Lua = id : () -> ()
        Fsharp = id : () -> ()
        Python = id : () -> ()
        Rust = id : () -> ()
        TypeScript = id : () -> ()
        Zig = id : () -> ()
    }

### test_trace_level

In [ ]:
inl test_trace_level level : bool =
    inl state = get_trace_state_or_init None
    inl level' = *state.level
    inl a : int = level |> severity
    inl b : int = level' |> severity
    if *state.enabled |> not
    then false
    else a >= b

In [1]:
///- --test
///> lua
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

test_trace_level Critical |> _assert_eq true
test_trace_level Verbose |> _assert_eq true
Debug |> set_trace_level
test_trace_level Verbose |> _assert_eq false
Verbose |> set_trace_level
test_trace_level Verbose |> _assert_eq true

.lua (lua)
__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = true }
.gleam (gleam)
__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = true }
.fsx (fsharp)
__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = true }
.py (cuda)
__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = true }
.rs (rust)
__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = true }
.ts (typescript)
__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = true }

0

### trace_raw

In [ ]:
inl trace_raw level fn =
    fun () =>
        inl trace_state =
            if level |> test_trace_level |> not
            then None
            else
                inl text = fn ()
                inl ({ count acc } & trace_state) = get_trace_state_or_init None
                fun () =>
                    count <- *count + 1
                |> exec_unit
                open rust
                open rust.rust_operators
                run_target_args (fun () => text, console.write_line) function
                    | Zig => fun _ => text |> console.write_line
                    | Rust => fun text, _ =>
                        global """
                            #[cfg(target_arch = "wasm32")]
                            fn spiral_trace_near_log(text: &str) {
                                use std::sync::atomic::{AtomicUsize, Ordering};
                                static LOGGED: AtomicUsize = AtomicUsize::new(0);
                                let cs: Vec<char> = text.chars().collect();
                                for c in cs.chunks(15000) {
                                    let s: String = c.iter().collect();
                                    let used = LOGGED.load(Ordering::Relaxed);
                                    let budget = 16000usize.saturating_sub(used);
                                    if budget < 13 {
                                        break;
                                    }
                                    let s = if s.len() <= budget {
                                        s
                                    } else {
                                        let mut end = budget - 12;
                                        while !s.is_char_boundary(end) {
                                            end -= 1;
                                        }
                                        format!("{} [truncated]", &s[..end])
                                    };
                                    LOGGED.store(used + s.len(), Ordering::Relaxed);
                                    near_sdk::env::log_str(&s);
                                }
                            }
                            """
                        if $'cfg\!(target_arch = "wasm32")' : bool
                        then
                            inl new_acc =
                                if *acc = ""
                                then text
                                elif text = ""
                                then *acc
                                else *acc ++\# text
                            inl chunks_len : i32 = !\\(new_acc, $'"(($0.chars().count() + 14999) / 15000) as i32"')
                            if text <>. "" && chunks_len <= 1i32
                            then acc <- new_acc
                            else
                                acc <- ""
                                !\\(new_acc, $'"{ \#[cfg(target_arch = \\\"wasm32\\\")] spiral_trace_near_log(&$0); }"')
                        else !\\(text, $'"println\!(\\\"{}\\\", $0)"')
                    | _ => fun text, write_line =>
                        text |> write_line
                text |> *trace_state.trace_file
                trace_state |> Some
        backend_switch {
            Gleam = fun () => $'let trace_state = fn () { !(trace_state |> optionm'.box) }' : ()
            Lua = id : () -> ()
            Fsharp = id : () -> ()
            Python = id : () -> ()
            Rust = id : () -> ()
            TypeScript = id : () -> ()
            Zig = id : () -> ()
        }
    |> exec_unit

### trace

In [ ]:
inl trace (level : trace_level) (text_fn : () -> string) (locals : () -> _) =
    fun () =>
        inl trace_state = get_trace_state_or_init None
        inl time =
            join
                run_target function
                    | Zig => fun () =>
                        global """extern "kernel32" fn GetLocalTime(t: *[8]u16) callconv(.winapi) void;"""
                        global """fn spiralClock() []const u8 {
    var secs: u64 = 0;
    if (builtin.os.tag == .windows) {
        var t: [8]u16 = undefined;
        GetLocalTime(&t);
        secs = @as(u64, t[4]) * 3600 + @as(u64, t[5]) * 60 + @as(u64, t[6]);
    } else if (builtin.os.tag == .linux) {
        var ts: std.os.linux.timespec = undefined;
        _ = std.os.linux.clock_gettime(.REALTIME, &ts);
        secs = @intCast(ts.sec);
    }
    return std.fmt.allocPrint(spiral_gpa, "{d:0>2}:{d:0>2}:{d:0>2}", .{ (secs / 3600) % 24, (secs / 60) % 60, secs % 60 }) catch @panic("out of memory");
}"""
                        $'spiralClock()' : string
                    | _ => fun () =>
                        run_target fun target =>
                            match target with
                            | Rust => fun () =>
                                open rust.rust_operators
                                inl seconds_of_day : u64 = $'{ \#[cfg(target_arch = "wasm32")] let (h, m, s) = { let secs = near_sdk::env::block_timestamp() / 1_000_000_000; ((secs / 3600) % 24, (secs / 60) % 60, secs % 60) }; \#[cfg(all(windows, not(target_arch = "wasm32")))] let (h, m, s) = { \#[repr(C)] struct St([u16; 8]); unsafe extern "system" { fn GetLocalTime(t: *mut St); } let mut t = St([0; 8]); unsafe { GetLocalTime(&mut t) }; (t.0[4] as u64, t.0[5] as u64, t.0[6] as u64) }; \#[cfg(all(unix, not(target_arch = "wasm32")))] let (h, m, s) = { \#[repr(C, align(8))] struct Tm([i32; 16]); unsafe extern "C" { fn localtime_r(t: *const std::os::raw::c_long, tm: *mut Tm) -> *mut Tm; } let secs = std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).map(|d| d.as_secs()).unwrap_or(0) as std::os::raw::c_long; let mut tm = Tm([0; 16]); unsafe { localtime_r(&secs, &mut tm) }; (tm.0[2] as u64, tm.0[1] as u64, tm.0[0] as u64) }; \#[cfg(not(any(windows, unix, target_arch = "wasm32")))] let (h, m, s) = { let secs = std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).map(|d| d.as_secs()).unwrap_or(0); ((secs / 3600) % 24, (secs / 60) % 60, secs % 60) }; h * 3600 + m * 60 + s }'
                                inl two_digits (n : u64) : string = (if n < 10u64 then "0" else "") ++# sm'.obj_to_string n
                                two_digits (seconds_of_day / 3600u64) ++# ":" ++# two_digits ((seconds_of_day / 60u64) % 60u64) ++# ":" ++# two_digits (seconds_of_day % 60u64)
                            | TypeScript => fun () =>
                                $'(() => { const d = new Date(); const p = (n: number) => String(n).padStart(2, "0"); return p(d.getHours()) + ":" + p(d.getMinutes()) + ":" + p(d.getSeconds()); })()' : string
                            | _ => fun () =>
                                match trace_state.repl_start |> optionm'.unbox with
                                | Some repl_start =>
                                    inl t =
                                        date_time.now ()
                                        |> date_time.to_timestamp
                                        |> fun (date_time.timestamp x) => x
                                        |> flip (-) repl_start
                                        |> date_time.time_span_of_micros
                                    date_time.date_time_milliseconds
                                        1i32 1i32 1i32
                                        (t |> date_time.hours)
                                        (t |> date_time.minutes)
                                        (t |> date_time.seconds)
                                        (t |> date_time.milliseconds)
                                | None => date_time.now ()
                                |> date_time.format (
                                    backend_switch {
                                        Gleam = fun () => "HH:mm:ss"
                                        Lua = fun () => "%H:%M:%S"
                                        Fsharp = fun () =>
                                            match target with
                                            | Rust => join "hh:mm:ss"
                                            | _ => join "HH:mm:ss"
                                        Python = fun () => "%H:%M:%S"
                                    }
                                )
        inl level_str =
            join
                inl color =
                    match level with
                    | Verbose => console.color_bright_black ()
                    | Debug => console.color_bright_blue ()
                    | Info => console.color_bright_green ()
                    | Warning => console.color_yellow ()
                    | Critical => console.color_bright_red ()
                inl level_str =
                    level
                    |> reflection.union_to_string
                    |> sm'.to_lower
                    |> sm'.index 0i32
                    |> sm'.format
                color ++# level_str ++# console.color_reset ()
        inl text = text_fn ()
        if text = ""
        then ""
        else
            inl locals = locals ()
            join
                inl count = *trace_state.count
                time ++# " "  ++#? count ++# level_str ++# " " ++# text ++# " / " ++#? locals
                |> fun x =>
                    join
                        x
                        |> sm'.trim_start []
                        |> sm'.trim_end [ ' '; '/' ]
    |> trace_raw level

In [2]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

trace Debug (fun () => "test1") id
trace Debug (fun () => "test2") id
get_trace_state_or_init None .count
|> fun x => *x
|> _assert_eq 3

.gleam (gleam)
15:08:52 1d test1
15:08:52 2d test2
__assert_eq { expected = 3 }
.lua (lua)
15:08:53 1d test1
15:08:53 2d test2
__assert_eq { expected = 3 }
.fsx (fsharp)
15:08:56 1d test1
15:08:56 2d test2
__assert_eq { expected = 3 }
.py (cuda)
15:08:57 1d test1
15:08:57 2d test2
__assert_eq { expected = 3 }
.rs (rust)
15:08:59 1d test1
15:08:59 2d test2
__assert_eq { expected = 3 }
.ts (typescript)
15:09:00 1d test1
15:09:00 2d test2
__assert_eq { expected = 3 }

0

### trace_format

In [ ]:
inl trace_format level fn obj =
    inl text, locals = fn obj
    inl locals = #?locals |> sm'.ellipsis_end 200
    trace level (fun () => text) (fun () => locals)
    obj

## main

In [3]:
///> fsharp

inl main () =
    init_trace_state None
    inl trace level text_fn (locals : () -> string) = trace level text_fn locals
    $'let trace x = !trace x' : ()